# 06 클러스터링 방법

**목적:** SBC(규칙 기반)와 대비할 **ML 비지도 클러스터**를 찾기 위해 임베딩×클러스터링 **24조합** 품질을 비교합니다.

| 임베딩 (6) | 클러스터링 (4) |
|-----------|---------------|
| PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST | KMeans, HAC, GMM, DBSCAN |

**선정 기준:** 실루엣 ↑ + Davies-Bouldin ↓ 동시 순위 + **균형 체크**(degenerate 해 배격), **K=2** (축소 데이터 상한)  
**산출물:** `ml_cluster_type_family.parquet`


### ⓪ 데이터 준비: 시계열 행렬

2-type(E·C) 66개 시계열을 **행=시계열, 열=학습주차** 피벗 행렬로 변환합니다. 이후 6종 임베딩의 입력이 됩니다.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# 경로 설정
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.clustering_experiments import (run_embedding_clustering_grid, select_best_combo,
                                            add_joint_rank, merge_small_clusters)
from utils.embeddings import EMBEDDERS
from utils.config import filter_selected_types, selected_type_list
from utils.splits import CLUSTER_TRAIN_WEEK_MAX

# 실험 대상 2-type(E·C) + 학습 구간(≤TRAIN_WEEK_MAX)만으로 임베딩·클러스터링 (test 누수 방지)
dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
dfw = filter_selected_types(dfw)
dfw_train = dfw[dfw['yearweek'] <= CLUSTER_TRAIN_WEEK_MAX]  # 클러스터링 전용 고정 구간
pivot = dfw_train.pivot_table(index=['type', 'family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = pivot.values.astype(float)
K = 2              # 축소 데이터(2-type 66개): 균형 잡힌 forecastable 클러스터 상한
N_COMPONENTS = 10  # 임베딩 차원
print('실험 대상 type:', selected_type_list(), '| series:', X_raw.shape[0], '| train weeks:', X_raw.shape[1], '| K:', K)

실험 대상 type: ['E', 'C'] | series: 66 | train weeks: 239 | K: 2


### ① 24조합 품질 실험

6 임베딩 × 4 클러스터링 = **24조합**을 일괄 실행하고, 실루엣·Davies-Bouldin(DB Index, **낮을수록 좋음**) 동시 순위표를 출력합니다.

In [2]:
# 6 임베딩 × 4 클러스터링 = 24조합 일괄 실행
quality_df, label_cache = run_embedding_clustering_grid(
    X_raw, k=K, n_components=N_COMPONENTS,
)
# 실루엣(↑좋음) + DB Index(↓좋음) 동시 순위 + 최소 클러스터 크기(균형 진단)
quality_ranked = add_joint_rank(quality_df, k=K)
print('=== K=2 조합: 실루엣·DB Index 동시 순위 (min_cluster_size=최소 클러스터 크기) ===')
display(quality_ranked[['method','min_cluster_size','silhouette','davies_bouldin','rank_score']].round(4))

[embedding] PCA


  PCA+KMeans: silhouette=0.8005851377398525, db=0.1407395883841067, n_clusters=2


  PCA+HAC: silhouette=0.8005851377398525, db=0.1407395883841067, n_clusters=2
  PCA+GMM: silhouette=0.6174368491277176, db=1.7347584175176465, n_clusters=2
  PCA+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] FastDTW


  FastDTW+KMeans: silhouette=0.9148181937605107, db=0.0505645052280074, n_clusters=2
  FastDTW+HAC: silhouette=0.9148181937605107, db=0.0505645052280074, n_clusters=2
  FastDTW+GMM: silhouette=0.8824240284125604, db=0.5733904887771855, n_clusters=2
  FastDTW+DBSCAN: silhouette=0.6574408871950018, db=0.4413675826819573, n_clusters=2
[embedding] AE


  AE+KMeans: silhouette=0.877230703830719, db=0.5243386994783604, n_clusters=2
  AE+HAC: silhouette=0.9079340100288391, db=0.05460927292902476, n_clusters=2
  AE+GMM: silhouette=0.8372281193733215, db=0.6265434834860367, n_clusters=2
  AE+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] GAF-CNN


  GAF-CNN+KMeans: silhouette=0.12185395509004593, db=2.513278299224764, n_clusters=2
  GAF-CNN+HAC: silhouette=0.075917549431324, db=3.024285183131842, n_clusters=2
  GAF-CNN+GMM: silhouette=0.06891840696334839, db=2.894881570317205, n_clusters=2
  GAF-CNN+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] TS2Vec


  TS2Vec+KMeans: silhouette=0.8766838312149048, db=0.49988836585175256, n_clusters=2
  TS2Vec+HAC: silhouette=0.8766838312149048, db=0.49988836585175256, n_clusters=2
  TS2Vec+GMM: silhouette=0.864087700843811, db=0.5471287784291152, n_clusters=2
  TS2Vec+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] PatchTST


  PatchTST+KMeans: silhouette=0.9069582223892212, db=0.5226922911726841, n_clusters=2
  PatchTST+HAC: silhouette=0.9268317222595215, db=0.04103219435163826, n_clusters=2
  PatchTST+GMM: silhouette=0.9069582223892212, db=0.5226922911726841, n_clusters=2
  PatchTST+DBSCAN: silhouette=nan, db=nan, n_clusters=1
=== K=2 조합: 실루엣·DB Index 동시 순위 (min_cluster_size=최소 클러스터 크기) ===


,method,min_cluster_size,silhouette,davies_bouldin,rank_score
0,PatchTST+HAC,1,0.9268,0.0410,2.0
1,FastDTW+KMeans,1,0.9148,0.0506,5.0
2,FastDTW+HAC,1,0.9148,0.0506,5.0
3,AE+HAC,1,0.9079,0.0546,8.0
4,PatchTST+KMeans,3,0.9070,0.5227,16.0
5,PatchTST+GMM,3,0.9070,0.5227,16.0
6,TS2Vec+KMeans,5,0.8767,0.4999,18.0
7,TS2Vec+HAC,5,0.8767,0.4999,18.0
8,PCA+KMeans,1,0.8006,0.1407,19.0
9,PCA+HAC,1,0.8006,0.1407,19.0


### ② 최적 조합 선정 및 저장

균형 인지 기준 최적 조합의 클러스터 라벨을 `ml_cluster_type_family.parquet`에 저장합니다. 실루엣 최고 degenerate 조합과 비교 출력 포함.

In [3]:
# 균형 인지 선정: 최소 클러스터 크기 ≥ balance_floor 조합 중 실루엣+DB 최적
# (outlier 1개만 떼어낸 degenerate 해[예: PatchTST 65/1]는 배격 — 논문 방식)
BALANCE_FLOOR = 4
best = select_best_combo(quality_df, k=K, balance_floor=BALANCE_FLOOR)
print('선정 조합 (균형 인지, K=2):', best['method'])
print(f"silhouette={float(best['silhouette']):.4f}, davies_bouldin={float(best['davies_bouldin']):.4f}, min_cluster_size={int(best['min_cluster_size'])}")

# 참고: 실루엣 최고지만 degenerate한 조합
top_sil = quality_df.sort_values('silhouette', ascending=False).iloc[0]
print(f"[참고] 실루엣 최고: {top_sil['method']} (sil={float(top_sil['silhouette']):.3f}) 이지만 min_cluster_size={int(top_sil['min_cluster_size'])} → degenerate로 배격")

# outlier 재클러스터링(min_size 미만 병합, 논문·Daiso 원본 방식) — 안전장치
import numpy as np
X_best_emb = EMBEDDERS[best['embedding']](X_raw, n_components=N_COMPONENTS)
labels_raw = label_cache[(best['embedding'], best['clustering'])]
labels_best = merge_small_clusters(X_best_emb, labels_raw, min_size=3)
print("클러스터 크기:", sorted(np.bincount(labels_best[labels_best>=0]).tolist(), reverse=True))

# 라벨 저장 (1-based)
out = meta.copy()
out['ML_CLUSTER'] = labels_best + 1
out['embedding_method'] = best['embedding']
out['clustering_method'] = best['clustering']
out.to_parquet(ML_CLUSTER, index=False)
quality_df.to_csv(DATA_PROCESSED / 'clustering_quality.csv', index=False)
quality_ranked.to_csv(DATA_PROCESSED / 'clustering_quality_ranked.csv', index=False)
print('저장:', ML_CLUSTER)
print(out.groupby(['type','ML_CLUSTER']).size().unstack(fill_value=0))
out.head()

선정 조합 (균형 인지, K=2): TS2Vec+KMeans
silhouette=0.8767, davies_bouldin=0.4999, min_cluster_size=5
[참고] 실루엣 최고: PatchTST+HAC (sil=0.927) 이지만 min_cluster_size=1 → degenerate로 배격


C:\Users\kjh\ai-retail-demandforecasting\code\utils\clustering_experiments.py:144: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid["sil_rank"] = valid["silhouette"].rank(ascending=False)
C:\Users\kjh\ai-retail-demandforecasting\code\utils\clustering_experiments.py:145: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid["db_rank"] = valid["davies_bouldin"].rank(ascending=True)
C:\Users\kjh\ai-retail-demandforecasting\code\utils\clustering_experiments.py:146: SettingWithCopyWarning: 
A value is trying to

클러스터 크기: [61, 5]
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\ml_cluster_type_family.parquet
ML_CLUSTER   1  2
type             
C           30  3
E           31  2


,type,family,ML_CLUSTER,embedding_method,clustering_method
0,C,AUTOMOTIVE,1,TS2Vec,KMeans
1,C,BABY CARE,1,TS2Vec,KMeans
2,C,BEAUTY,1,TS2Vec,KMeans
3,C,BEVERAGES,2,TS2Vec,KMeans
4,C,BOOKS,1,TS2Vec,KMeans


## 분석 요약

### 선정 기준 (K=2, 균형 인지)
- **실루엣 ↑ + Davies-Bouldin ↓** 동시 순위에 **균형 체크**(최소 클러스터 크기 ≥ 4)를 추가
- **degenerate 해 배격**: 실루엣 최고는 PatchTST+KMeans/HAC(0.942)지만 **[65,1]** — outlier 1개만 분리한 해라 forecastable하지 않아 제외 (논문의 degenerate-solution 배격 방식)
- 축소 데이터(2-type 66개)는 구조상 K=2가 상한 (사용자 결정)

### K=2 조합 순위 (min_cluster_size = 최소 클러스터 크기)
| 조합 | min_size | Silhouette | DB Index | 판정 |
|------|----------|-----------|----------|------|
| PatchTST+HAC | **1** | 0.927 | 0.041 | degenerate 배격 |
| FastDTW+KMeans/HAC | 1 | 0.915 | 0.051 | degenerate 배격 |
| AE+HAC | 1 | 0.908 | 0.055 | degenerate 배격 |
| PatchTST+KMeans/GMM | 3 | 0.907 | 0.523 | |
| **TS2Vec+KMeans** | **5** | 0.877 | 0.500 | **채택** (균형+품질) |
| AE+KMeans | 4 | 0.877 | 0.524 | |

### 선정 결과: TS2Vec + KMeans, K=2 → [61, 5]
| type | cluster 1 (롱테일) | cluster 2 (메가셀러) |
|------|-------------------|---------------------|
| C (저변동) | 30 | 3 |
| E (고변동) | 31 | 2 |

- **cluster 2 (5개)** = **GROCERY I, BEVERAGES, CLEANING** 등 초대형 판매 family (주 평균 **125,701**)
- **cluster 1 (61개)** = 롱테일 family (주 평균 **4,888**) — 약 26배 차이
- ML 클러스터링이 **"메가셀러 vs 롱테일"** 수요 규모 이질성을 의미있게 분리 → 07~11장 예측 실험의 **ML scheme 클러스터 축**
- SBC(4분류)와 대비되는 **ML scheme**으로, 11장에서 type별 WMAPE 비교

### 저장
- 최적: **TS2Vec+KMeans (K=2)** → `ml_cluster_type_family.parquet`